In [1]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ==============================
# 1. IMPORTS
# ==============================
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# 2. LOAD DATA
# ==============================
movies = pd.read_csv('../data/raw/movies.csv')

# ==============================
# 3. CLEAN DATA
# ==============================
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
movies = movies[['movieId', 'title', 'genres']]

# Handle missing values
movies['genres'] = movies['genres'].fillna('')

# ==============================
# 4. TF-IDF VECTORIZATION
# ==============================
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# ==============================
# 5. COSINE SIMILARITY
# ==============================
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# ==============================
# 6. CREATE INDEX MAPPING
# ==============================
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# ==============================
# 7. RECOMMENDATION FUNCTION
# ==============================
def recommend(movie_title, top_n=5):
    if movie_title not in indices:
        return "Movie not found"
    
    idx = indices[movie_title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    return movies['title'].iloc[movie_indices]

# ==============================
# 8. TEST
# ==============================
print(recommend("Toy Story (1995)", top_n=5))

1                        Jumanji (1995)
7                   Tom and Huck (1995)
4    Father of the Bride Part II (1995)
2               Grumpier Old Men (1995)
6                        Sabrina (1995)
Name: title, dtype: str


In [5]:
import os
print(os.getcwd())
print(os.listdir('../data/raw/'))

d:\PROJECTS\Movie reccomdation system\movie-recommender\notebooks
['movies.csv', 'ratings.csv']
